In [1]:
import os
import glob
import numpy as np
from scipy.interpolate import make_interp_spline
import subprocess
import netCDF4 as nc
from tqdm import tqdm


# ============================================================
# Helpers
# ============================================================

def find_file(pattern: str) -> str:
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Cannot find file matching pattern: {pattern}")
    return files[0]


def unique_within_tolerance(arr, tol):
    arr = np.asarray(arr)
    sorted_arr = np.sort(arr)
    unique = [sorted_arr[0]]
    for i in range(1, len(sorted_arr)):
        if np.abs(sorted_arr[i] - unique[-1]) > tol:
            unique.append(sorted_arr[i])
    return np.array(unique)


def cleanup_old_outputs():
    patterns = [
        "*.e",
        "*.e-s*",
        "*.cp",
        "*.cpr",
        "*.csv",
        "*.json",
        "*.xda",
        "*.xdr",
        "*.xmf",
        "*.exo",
        "*.out",
        "*.log",
    ]
    for pat in patterns:
        for f in glob.glob(pat):
            try:
                os.remove(f)
            except OSError:
                pass


def print_dataset_variables(file_path):
    dataset = nc.Dataset(file_path, "r")
    print(f"\nVariables in {file_path}:")
    for k in dataset.variables.keys():
        print(" ", k, dataset.variables[k].dimensions, dataset.variables[k].shape)
    dataset.close()


def decode_name_array(arr):
    """Decode Exodus char array names if present."""
    out = []
    arr = np.asarray(arr)
    if arr.dtype.kind in ("S", "U"):
        if arr.ndim == 2:
            for row in arr:
                s = b"".join(row).decode("utf-8", errors="ignore").strip() if row.dtype.kind == "S" else "".join(row).strip()
                out.append(s)
        else:
            out = [str(x).strip() for x in arr]
    else:
        for row in arr:
            chars = []
            for x in row:
                if isinstance(x, bytes):
                    chars.append(x.decode("utf-8", errors="ignore"))
                else:
                    chars.append(chr(x) if isinstance(x, (int, np.integer)) else str(x))
            out.append("".join(chars).strip())
    return out


# ============================================================
# Exodus variable-name helpers
# ============================================================

def get_exodus_var_names(dataset, kind="elem"):
    """
    kind = 'elem' or 'nod'
    Returns decoded variable names from Exodus if present.
    """
    candidates = []
    if kind == "elem":
        candidates = ["name_elem_var", "elem_var_names"]
    elif kind == "nod":
        candidates = ["name_nod_var", "nod_var_names"]

    for c in candidates:
        if c in dataset.variables:
            return decode_name_array(dataset.variables[c][:])

    return []


def match_exodus_variable(dataset, candidates, kind="elem", block_suffix="eb1"):
    """
    Find an Exodus variable by matching against variable names first,
    then by fallback index-based search.
    """
    keys = list(dataset.variables.keys())

    if kind == "elem":
        name_list = get_exodus_var_names(dataset, kind="elem")
        for idx, nm in enumerate(name_list, start=1):
            low = nm.lower()
            if any(c in low for c in candidates):
                key = f"vals_elem_var{idx}{block_suffix}"
                if key in dataset.variables:
                    return key

        # fallback: try direct key name matches
        elem_vars = sorted([k for k in keys if "vals_elem_var" in k.lower() and block_suffix in k.lower()])
        return elem_vars

    if kind == "nod":
        name_list = get_exodus_var_names(dataset, kind="nod")
        for idx, nm in enumerate(name_list, start=1):
            low = nm.lower()
            if any(c in low for c in candidates):
                key = f"vals_nod_var{idx}"
                if key in dataset.variables:
                    return key

        nod_vars = sorted([k for k in keys if "vals_nod_var" in k.lower()])
        return nod_vars

    raise ValueError("kind must be 'elem' or 'nod'")


# ============================================================
# Exodus readers
# ============================================================

def read_neu_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    x_coords = dataset.variables["coordx"][:]
    y_coords = dataset.variables["coordy"][:]

    # Try to find nodal variable named u first
    nod_match = match_exodus_variable(dataset, candidates=["u"], kind="nod")
    if isinstance(nod_match, list):
        if not nod_match:
            raise RuntimeError("Cannot find any nodal variable in neutron Exodus file.")
        var_name = nod_match[0]
    else:
        var_name = nod_match

    u = dataset.variables[var_name][:]

    unique_x = unique_within_tolerance(np.array(x_coords), 1e-6)
    unique_y = unique_within_tolerance(np.array(y_coords), 1e-3)

    time_steps = u.shape[0]
    z_matrix = np.zeros((time_steps, len(unique_x), len(unique_y)))
    mask = np.zeros((len(unique_x), len(unique_y)))

    for i in range(len(x_coords)):
        xi = np.argmin(np.abs(x_coords[i] - unique_x))
        yi = np.argmin(np.abs(y_coords[i] - unique_y))
        mask[xi, yi] = 1
        z_matrix[:, xi, yi] = u[:, i]

    # nodal -> cell-center average
    z_matrix = (z_matrix[..., :-1] + z_matrix[..., 1:]) / 2
    z_matrix = (z_matrix[:, :-1] + z_matrix[:, 1:]) / 2

    assert np.mean(mask) == 1, "Neutron mapping incomplete"
    dataset.close()
    return z_matrix


def read_fuel_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    x_coords = dataset.variables["coordx"][:]
    y_coords = dataset.variables["coordy"][:]

    # Solid temperature: usually first nodal var, but try match name T
    nod_match = match_exodus_variable(dataset, candidates=["t", "temp", "temperature"], kind="nod")
    if isinstance(nod_match, list):
        if not nod_match:
            raise RuntimeError("Cannot find solid nodal temperature variable.")
        temp_key = nod_match[0]
    else:
        temp_key = nod_match
    u = dataset.variables[temp_key][:]

    # Solid flux: elemental variable, try to match flux
    elem_match = match_exodus_variable(dataset, candidates=["flux"], kind="elem", block_suffix="eb1")
    if isinstance(elem_match, list):
        if not elem_match:
            raise RuntimeError("Cannot find solid elemental flux variable.")
        flux_key = elem_match[0]
    else:
        flux_key = elem_match

    flux = np.asarray(dataset.variables[flux_key][:])

    unique_x = unique_within_tolerance(np.array(x_coords), 1e-6)
    unique_y = unique_within_tolerance(np.array(y_coords), 1e-3)

    time_steps = u.shape[0]
    z_matrix = np.zeros((1, time_steps, len(unique_x), len(unique_y)))
    mask = np.zeros((len(unique_x), len(unique_y)))

    for i in range(len(x_coords)):
        xi = np.argmin(np.abs(x_coords[i] - unique_x))
        yi = np.argmin(np.abs(y_coords[i] - unique_y))
        mask[xi, yi] = 1
        z_matrix[0, :, xi, yi] = u[:, i]

    assert np.mean(mask) == 1, "Solid mapping incomplete"

    # nodal -> cell-center average
    z_matrix = (z_matrix[:, :, 1:, 1:] + z_matrix[:, :, :-1, :-1]) / 2

    if flux.ndim != 2:
        raise RuntimeError(f"Unexpected solid flux shape: {flux.shape}")

    num_time_steps = flux.shape[0]

    # solid mesh = 8 x 64 cells
    flux = flux.reshape(1, num_time_steps, 64, 8).transpose(0, 1, 3, 2)

    dataset.close()
    return np.concatenate((z_matrix, flux), axis=0)
def read_fluid_to_np(file_path):
    dataset = nc.Dataset(file_path, "r")

    T_fluid = np.asarray(dataset.variables["vals_elem_var1eb1"][:])
    pressure = np.asarray(dataset.variables["vals_elem_var3eb1"][:])
    vel_x = np.asarray(dataset.variables["vals_elem_var4eb1"][:])
    vel_y = np.asarray(dataset.variables["vals_elem_var5eb1"][:])

    num_time = T_fluid.shape[0]

    T_fluid = T_fluid.reshape(1, num_time, 64, 12)
    pressure = pressure.reshape(1, num_time, 64, 12)
    vel_x = vel_x.reshape(1, num_time, 64, 12)
    vel_y = vel_y.reshape(1, num_time, 64, 12)

    dataset.close()
    return np.concatenate((T_fluid, pressure, vel_x, vel_y), axis=0).transpose(0, 1, 3, 2)


# ============================================================
# Generate phi BC values (65 x 16)
# ============================================================

def gen_phi_BC():
    def generate_x_coords(min_val, max_val, n_points, threshold):
        x_coords = []
        while len(x_coords) < n_points:
            x = np.random.uniform(min_val, max_val)
            if (
                (len(x_coords) == 0 or all(abs(x - xi) > threshold for xi in x_coords))
                and abs(x - min_val) > threshold
                and abs(x - max_val) > threshold
            ):
                x_coords.append(x)
        return sorted(x_coords)

    phi_all = []
    time_steps = 16

    while True:
        x_random = generate_x_coords(0, 0.75, 5, 0.075)
        y_random = np.random.uniform(0.5, 3.0, 5)

        x_fixed = np.array([0.0, 0.75])
        y_fixed = np.array([0.5, 0.5])

        x_all = np.concatenate(([x_fixed[0]], x_random, [x_fixed[1]]))
        y_all = np.concatenate(([y_fixed[0]], y_random, [y_fixed[1]]))

        spline = make_interp_spline(x_all, y_all)
        x_spline = np.linspace(0, 0.75, 65)
        y_spline = spline(x_spline)

        if np.all(y_spline >= 0):
            break

    phi_all.append(y_spline)

    max_increase = np.random.uniform(0.1, 0.9)
    max_index = np.argmax(y_random)

    for _ in range(1, time_steps):
        peak_increase = np.random.uniform(0.1, max_increase)
        factor = np.random.uniform(0.1, 0.7, 5)
        y_random += factor * peak_increase
        y_random[max_index] += (1 - factor[max_index]) * peak_increase

        y_all = np.concatenate(([y_fixed[0]], y_random, [y_fixed[1]]))
        spline = make_interp_spline(x_all, y_all)
        y_spline = np.abs(spline(x_spline))
        phi_all.append(np.abs(y_spline))

    return np.array(phi_all).transpose(1, 0)


# ============================================================
# Build MOOSE PiecewiseMultilinear phi.txt from phi_base.txt
# ============================================================

def replacements(function, Lx=0.0076, Ly=0.75, Lt=5, nx=8, ny=64, nt=16,
                 bias_x=0, bias_y=0, bias_t=0):
    dx = Lx / nx
    dy = Ly / ny
    dt = Lt / nt

    coor_y_str = "%.5f" % bias_y
    coor_t_str = ""

    for i in range(ny):
        coor_y_str += " %.5f" % (dy * (i + 1) + bias_y)
    for i in range(nt):
        coor_t_str += "%.5f " % (dt * (i + 1) + bias_t)

    y_values = np.array(list(map(float, coor_y_str.split())))
    Z = function()  # shape (65, 16)

    if Z.shape != (len(y_values), nt):
        raise RuntimeError(
            f"phi_BC shape mismatch: got {Z.shape}, expected ({len(y_values)}, {nt})"
        )

    data = ""
    for i in range(nt):
        for j in range(len(y_values)):
            data += "%.6f " % Z[j, i]

    repl = {
        "y_coor": coor_y_str,
        "t_coor": coor_t_str,
        "data": data
    }
    return repl, Z


def write_inp(base_file, out_file, repl):
    with open(base_file, "r", encoding="utf-8") as base:
        template = base.read()
    src = template % repl
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(src)


def ensure_phi_base_exists():
    if not os.path.exists("phi_base.txt"):
        with open("phi_base.txt", "w", encoding="utf-8") as f:
            f.write(
                "AXIS Y\n"
                "%(y_coor)s\n\n"
                "AXIS T\n"
                "%(t_coor)s\n\n"
                "DATA\n"
                "%(data)s\n"
            )


# ============================================================
# Run MOOSE
# ============================================================

def run_moose(input_file="solid_pinn_ref.i", executable="../../workspace-opt"):
    command = [executable, "-i", input_file]
    result = subprocess.run(command, text=True)
    print("Return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("MOOSE run failed")

    case_base = os.path.splitext(os.path.basename(input_file))[0]
    return case_base


# ============================================================
# Data collection
# ============================================================

def collect_case_data(case_base, debug_print_vars=False):
    solid_file = f"./{case_base}_exodus.e"
    neutron_file = f"./{case_base}_out_sub_app0_exodus.e"
    fluid_file = f"./{case_base}_out_sub_app0_sub_app0_exodus.e"

    for f in [solid_file, neutron_file, fluid_file]:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Missing expected output file: {f}")

    if debug_print_vars:
        print("solid_file  :", solid_file)
        print("neutron_file:", neutron_file)
        print("fluid_file  :", fluid_file)
        print_dataset_variables(solid_file)
        print_dataset_variables(neutron_file)
        print_dataset_variables(fluid_file)

    Tfuel = read_fuel_to_np(solid_file)
    fluid = read_fluid_to_np(fluid_file)
    neutron = read_neu_to_np(neutron_file)

    return Tfuel, fluid, neutron

# ============================================================
# Main
# ============================================================

def main(n=2, debug_print_vars=False, input_file="solid_pinn_ref.i"):
    ensure_phi_base_exists()

    phiBC_all = []
    T_fuel_all = []
    T_fluid_all = []
    neu_all = []

    for _ in tqdm(range(n), desc="Generating coupled cases", total=n):
        cleanup_old_outputs()

        repl, phi = replacements(
            function=gen_phi_BC,
            Lx=0.0076,
            Ly=0.75,
            Lt=5,
            nx=8,
            ny=64,
            nt=16,
            bias_x=0,
            bias_y=0,
            bias_t=0,
        )
        write_inp("phi_base.txt", "phi.txt", repl)

        case_base = run_moose(input_file=input_file)
        Tfuel, fluid, neutron = collect_case_data(case_base, debug_print_vars=debug_print_vars)

        phiBC_all.append(phi)
        T_fuel_all.append(Tfuel)
        T_fluid_all.append(fluid)
        neu_all.append(neutron)

    os.makedirs("./output", exist_ok=True)
    np.save("./output/phiBC_to_phi.npy", np.array(phiBC_all))
    np.save("./output/nft_Tfuel.npy", np.array(T_fuel_all))
    np.save("./output/nft_Tfluid.npy", np.array(T_fluid_all))
    np.save("./output/nft_phi.npy", np.array(neu_all))

    print("\nSaved:")
    print("  ./output/phiBC_to_phi.npy")
    print("  ./output/nft_Tfuel.npy")
    print("  ./output/nft_Tfluid.npy")
    print("  ./output/nft_phi.npy")


def main1(debug_print_vars=False, input_file="solid_pinn_ref.i"):
    ensure_phi_base_exists()
    cleanup_old_outputs()

    repl, phi = replacements(
        function=gen_phi_BC,
        Lx=0.0076,
        Ly=0.75,
        Lt=5,
        nx=8,
        ny=64,
        nt=16,
        bias_x=0,
        bias_y=0,
        bias_t=0,
    )
    write_inp("phi_base.txt", "phi.txt", repl)

    case_base = run_moose(input_file=input_file)
    Tfuel, fluid, neutron = collect_case_data(case_base, debug_print_vars=debug_print_vars)

    os.makedirs("./output", exist_ok=True)
    np.save("./output/phiBC_to_phi.npy", np.array([phi]))
    np.save("./output/nft_Tfuel.npy", np.array([Tfuel]))
    np.save("./output/nft_Tfluid.npy", np.array([fluid]))
    np.save("./output/nft_phi.npy", np.array([neutron]))

    print("\nSingle case saved.")
    print("phi shape       :", np.array([phi]).shape)
    print("Tfuel shape     :", np.array([Tfuel]).shape)
    print("Tfluid shape    :", np.array([fluid]).shape)
    print("neutron shape   :", np.array([neutron]).shape)


# ============================================================
# Jupyter-safe execution
# ============================================================

if __name__ == "__main__":
    main1(debug_print_vars=True, input_file="solid_pinn_ref.i")

/usr/bin/ld: cannot find -lmpi_cxx: No such file or directory
/usr/bin/ld: cannot find -lmpi: No such file or directory
/usr/bin/ld: cannot find -lopen-rte: No such file or directory
/usr/bin/ld: cannot find -lopen-pal: No such file or directory
/usr/bin/ld: cannot find -lhwloc: No such file or directory
/usr/bin/ld: cannot find -levent_core: No such file or directory
/usr/bin/ld: cannot find -levent_pthreads: No such file or directory
collect2: error: ld returned 1 exit status
JIT compile failed.


/usr/bin/ld: cannot find -lmpi_cxx: No such file or directory
/usr/bin/ld: cannot find -lmpi: No such file or directory
/usr/bin/ld: cannot find -lopen-rte: No such file or directory
/usr/bin/ld: cannot find -lopen-pal: No such file or directory
/usr/bin/ld: cannot find -lhwloc: No such file or directory
/usr/bin/ld: cannot find -levent_core: No such file or directory
/usr/bin/ld: cannot find -levent_pthreads: No such file or directory
collect2: error: ld returned 1 exit status
JIT compile failed.


/usr/bin/ld: cannot find -lmpi_cxx: No such file or directory
/usr/bin/ld: cannot find -lmpi: No such file or directory
/usr/bin/ld: cannot find -lopen-rte: No such file or directory
/usr/bin/ld: cannot find -lopen-pal: No such file or directory
/usr/bin/ld: cannot find -lhwloc: No such file or directory
/usr/bin/ld: cannot find -levent_core: No such file or directory
/usr/bin/ld: cannot find -levent_pthreads: No such file or directory
collect2: error: ld returned 1 exit status


JIT compile failed.


/usr/bin/ld: cannot find -lmpi_cxx: No such file or directory
/usr/bin/ld: cannot find -lmpi: No such file or directory
/usr/bin/ld: cannot find -lopen-rte: No such file or directory
/usr/bin/ld: cannot find -lopen-pal: No such file or directory
/usr/bin/ld: cannot find -lhwloc: No such file or directory
/usr/bin/ld: cannot find -levent_core: No such file or directory
/usr/bin/ld: cannot find -levent_pthreads: No such file or directory
collect2: error: ld returned 1 exit status
JIT compile failed.


...

JIT compile failed.


/usr/bin/ld: cannot find -lmpi_cxx: No such file or directory
/usr/bin/ld: cannot find -lmpi: No such file or directory
/usr/bin/ld: cannot find -lopen-rte: No such file or directory
/usr/bin/ld: cannot find -lopen-pal: No such file or directory
/usr/bin/ld: cannot find -lhwloc: No such file or directory
/usr/bin/ld: cannot find -levent_core: No such file or directory
/usr/bin/ld: cannot find -levent_pthreads: No such file or directory
collect2: error: ld returned 1 exit status
JIT compile failed.


/usr/bin/ld: cannot find -lmpi_cxx: No such file or directory
/usr/bin/ld: cannot find -lmpi: No such file or directory
/usr/bin/ld: cannot find -lopen-rte: No such file or directory
/usr/bin/ld: cannot find -lopen-pal: No such file or directory
/usr/bin/ld: cannot find -lhwloc: No such file or directory
/usr/bin/ld: cannot find -levent_core: No such file or directory
/usr/bin/ld: cannot find -levent_pthreads: No such file or directory
collect2: error: ld returned 1 exit status
JIT compile failed.


  Finished Instantiating Sub-Apps                                                        [ 12.31 s] [  244 MB]
Finished Setting Up                                                                      [ 12.65 s] [  246 MB]
Framework Information:
MOOSE Version:           git commit aa3cb529c3 on 2026-03-11
LibMesh Version:         
PETSc Version:           3.24.4
SLEPc Version:           3.24.0
Current Time:            Fri Jul 10 17:52:09 2026
Executable Timestamp:    Thu Mar 12 19:57:58 2026

Input File(s):
  /projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/solid_pinn_ref.i

Checkpoint:
  Wall Time Interval:      Every 3600 s
  User Checkpoint:         Disabled
  # Checkpoints Kept:      2
  Execute On:              TIMESTEP_END 

Parallelism:
  Num Processors:          1
  Num Threads:             1

Mesh: 
  Parallel Type:           replicated
  Mesh Dimension:          2
  Spatial Dimension:       2
  Nodes:                   585
  Elems:                   512
  Num Subdomain

sub_app0_sub_app0: 
sub_app0_sub_app0: Time Step 0, time = 0


In [15]:
print(fluid.shape)
print(fluid[0].min(), fluid[0].max())
print(fluid[1].min(), fluid[1].max())
print(fluid[2].min(), fluid[2].max())
print(fluid[3].min(), fluid[3].max())

(4, 17, 12, 64)
559.9453571841821 691.8876639452255
0.0 4234.93880240171
-2.6865901244236703e-14 1.4197070672434489e-14
0.0 0.3999999999944455


In [7]:
import netCDF4 as nc
import numpy as np

# your loaded p_true from npy
# p_true shape should be (17, 12, 64)

f = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/solid_pinn_ref_out_sub_app0_sub_app0_exodus.e"
ds = nc.Dataset(f)

# pressure is vals_elem_var3eb1 from your verified element names
p_raw = np.asarray(ds.variables["vals_elem_var3eb1"][:])   # (17, 768)
Nt = p_raw.shape[0]

# reshape exactly like your exporter
p_from_exodus = p_raw.reshape(1, Nt, 64, 12).transpose(0, 1, 3, 2)[0]   # (17, 12, 64)

ds.close()

print("p_from_exodus shape:", p_from_exodus.shape)
print("p_from_exodus min/max:", p_from_exodus.min(), p_from_exodus.max())

print("p_true shape:", p_true.shape)
print("p_true min/max:", p_true.min(), p_true.max())

print("difference mean abs:", np.mean(np.abs(p_from_exodus - p_true)))
print("difference max abs :", np.max(np.abs(p_from_exodus - p_true)))

p_from_exodus shape: (17, 12, 64)
p_from_exodus min/max: 0.0 4234.93880240171


NameError: name 'p_true' is not defined

In [8]:
case_base = "solid_pinn_ref"
Tfuel, fluid, neutron = collect_case_data(case_base, debug_print_vars=False)

print("fluid shape:", fluid.shape)
print("Tf:", fluid[0].min(), fluid[0].max())
print("p :", fluid[1].min(), fluid[1].max())
print("u :", fluid[2].min(), fluid[2].max())
print("v :", fluid[3].min(), fluid[3].max())

fluid shape: (4, 17, 12, 64)
Tf: 559.9453571841821 691.8876639452255
p : 0.0 4234.93880240171
u : -2.6865901244236703e-14 1.4197070672434489e-14
v : 0.0 0.3999999999944455


In [9]:
import glob
import os

files = sorted(glob.glob("/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/*.e*"))
for f in files:
    print(os.path.basename(f))

solid_pinn_ref_exodus.e
solid_pinn_ref_out_sub_app0_exodus.e
solid_pinn_ref_out_sub_app0_sub_app0_exodus.e


In [19]:
import netCDF4 as nc
import numpy as np

def decode_name_array(arr):
    out = []
    arr = np.asarray(arr)
    if arr.dtype.kind in ("S", "U"):
        if arr.ndim == 2:
            for row in arr:
                s = b"".join(row).decode("utf-8", errors="ignore").strip() if row.dtype.kind == "S" else "".join(row).strip()
                out.append(s)
        else:
            out = [str(x).strip() for x in arr]
    else:
        for row in arr:
            chars = []
            for x in row:
                if isinstance(x, bytes):
                    chars.append(x.decode("utf-8", errors="ignore"))
                else:
                    chars.append(chr(x) if isinstance(x, (int, np.integer)) else str(x))
            out.append("".join(chars).strip())
    return out

f = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/solid_pinn_ref_out_sub_app0_exodus.e"
ds = nc.Dataset(f)

print("time_whole =", ds.variables["time_whole"][:])

if "name_nod_var" in ds.variables:
    print("nodal names =", decode_name_array(ds.variables["name_nod_var"][:]))

if "name_elem_var" in ds.variables:
    print("element names =", decode_name_array(ds.variables["name_elem_var"][:]))

for k in ds.variables:
    print(k, ds.variables[k].dimensions, ds.variables[k].shape)

time_whole = [0.     0.3125 0.625  0.9375 1.25   1.5625 1.875  2.1875 2.5    2.8125
 3.125  3.4375 3.75   4.0625 4.375  4.6875 5.    ]
nodal names = ['T', 'aux_sigma_af', 'u']
element names = ['flux']
time_whole ('time_step',) (17,)
eb_status ('num_el_blk',) (2,)
eb_prop1 ('num_el_blk',) (2,)
ns_status ('num_node_sets',) (4,)
ns_prop1 ('num_node_sets',) (4,)
ss_status ('num_side_sets',) (4,)
ss_prop1 ('num_side_sets',) (4,)
coordx ('num_nodes',) (1365,)
coordy ('num_nodes',) (1365,)
eb_names ('num_el_blk', 'len_name') (2, 256)
ns_names ('num_node_sets', 'len_name') (4, 256)
ss_names ('num_side_sets', 'len_name') (4, 256)
coor_names ('num_dim', 'len_name') (2, 256)
node_num_map ('num_nodes',) (1365,)
connect1 ('num_el_in_blk1', 'num_nod_per_el1') (512, 4)
connect2 ('num_el_in_blk2', 'num_nod_per_el2') (768, 4)
elem_num_map ('num_elem',) (1280,)
elem_ss1 ('num_side_ss1',) (20,)
side_ss1 ('num_side_ss1',) (20,)
elem_ss2 ('num_side_ss2',) (64,)
side_ss2 ('num_side_ss2',) (64,)
elem_ss3 ('n

In [22]:
u = ds.variables["vals_nod_var3"][:]   # change index if needed after checking names
x = ds.variables["coordx"][:]
y = ds.variables["coordy"][:]
t = ds.variables["time_whole"][:]

print("u shape:", u.shape)
print("first saved time:", t[0])
print("u[0] min/max:", u[0].min(), u[0].max())

left_mask   = np.isclose(x, 0.0)
right_mask  = np.isclose(x, 0.019)
bottom_mask = np.isclose(y, 0.0)
top_mask    = np.isclose(y, 0.75)

print("left  first slice min/max:", u[0, left_mask].min(), u[0, left_mask].max())
print("right first slice min/max:", u[0, right_mask].min(), u[0, right_mask].max())
print("bottom first slice min/max:", u[0, bottom_mask].min(), u[0, bottom_mask].max())
print("top    first slice min/max:", u[0, top_mask].min(), u[0, top_mask].max())

u shape: (17, 1365)
first saved time: 0.0
u[0] min/max: 0.0015926534214665267 2.0
left  first slice min/max: 0.0015926534214665267 2.0
right first slice min/max: 0.0015926534214665267 2.0
bottom first slice min/max: 0.0015926534214665267 0.0015926534214665267
top    first slice min/max: 0.0015926534214674149 0.0015926534214674149


In [23]:
x_unique = np.unique(np.round(x, 10))
x_mid = x_unique[len(x_unique)//2]
mid_mask = np.isclose(x, x_mid)

yy = y[mid_mask]
uu = u[0, mid_mask]

order = np.argsort(yy)
yy = yy[order]
uu = uu[order]

u_ic_expected = 2.0 * np.cos(3.14 * (yy - 0.375) / 0.75)

print("IC mean abs error:", np.mean(np.abs(uu - u_ic_expected)))
print("IC max  abs error:", np.max(np.abs(uu - u_ic_expected)))
print("u[0] along mid-x min/max:", uu.min(), uu.max())
print("expected min/max:", u_ic_expected.min(), u_ic_expected.max())

IC mean abs error: 1.588472942925224e-16
IC max  abs error: 8.881784197001252e-16
u[0] along mid-x min/max: 0.0015926534214665267 2.0
expected min/max: 0.0015926534214665267 2.0


In [26]:
import netCDF4 as nc
import numpy as np

def decode_name_array(arr):
    out = []
    arr = np.asarray(arr)
    if arr.dtype.kind in ("S", "U"):
        if arr.ndim == 2:
            for row in arr:
                s = b"".join(row).decode("utf-8", errors="ignore").strip() if row.dtype.kind == "S" else "".join(row).strip()
                out.append(s)
        else:
            out = [str(x).strip() for x in arr]
    else:
        for row in arr:
            chars = []
            for x in row:
                if isinstance(x, bytes):
                    chars.append(x.decode("utf-8", errors="ignore"))
                else:
                    chars.append(chr(x) if isinstance(x, (int, np.integer)) else str(x))
            out.append("".join(chars).strip())
    return out

f = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/solid_pinn_ref_out_sub_app0_exodus.e"
ds = nc.Dataset(f)

print("time_whole =", ds.variables["time_whole"][:])

nod_names = decode_name_array(ds.variables["name_nod_var"][:])
print("nodal names =", nod_names)

# phi is nodal variable named 'u'
phi_idx = nod_names.index("u") + 1
phi = ds.variables[f"vals_nod_var{phi_idx}"][:]

x = ds.variables["coordx"][:]
y = ds.variables["coordy"][:]
t = ds.variables["time_whole"][:]

# check BCs at non-initial time
k = 1   # t = 0.3125
print("\nchecking time =", t[k])

left_mask   = np.isclose(x, 0.0)
right_mask  = np.isclose(x, 0.019)
bottom_mask = np.isclose(y, 0.0)
top_mask    = np.isclose(y, 0.75)

print("\nLEFT boundary (from phi.txt)")
print("mean abs:", np.mean(np.abs(phi[k, left_mask])))
print("min/max :", phi[k, left_mask].min(), phi[k, left_mask].max())

print("\nRIGHT boundary (should be 0)")
print("mean abs:", np.mean(np.abs(phi[k, right_mask])))
print("max abs :", np.max(np.abs(phi[k, right_mask])))
print("min/max :", phi[k, right_mask].min(), phi[k, right_mask].max())

print("\nBOTTOM boundary (should be 0.5)")
print("mean abs error:", np.mean(np.abs(phi[k, bottom_mask] - 0.5)))
print("max abs error :", np.max(np.abs(phi[k, bottom_mask] - 0.5)))
print("min/max       :", phi[k, bottom_mask].min(), phi[k, bottom_mask].max())

print("\nTOP boundary (should be 0.5)")
print("mean abs error:", np.mean(np.abs(phi[k, top_mask] - 0.5)))
print("max abs error :", np.max(np.abs(phi[k, top_mask] - 0.5)))
print("min/max       :", phi[k, top_mask].min(), phi[k, top_mask].max())

# optional: check IC at t=0
yy = np.unique(np.round(y, 10))
yy.sort()
phi_ic_expected = 2.0 * np.cos(3.14 * (yy - 0.375) / 0.75)

# pick one interior x-line
x_unique = np.unique(np.round(x, 10))
x_unique.sort()
x_mid = x_unique[len(x_unique)//2]
mid_mask = np.isclose(x, x_mid)

yy_mid = y[mid_mask]
phi_t0_mid = phi[0, mid_mask]

order = np.argsort(yy_mid)
yy_mid = yy_mid[order]
phi_t0_mid = phi_t0_mid[order]

phi_ic_expected_mid = 2.0 * np.cos(3.14 * (yy_mid - 0.375) / 0.75)

print("\nIC check at t=0")
print("mean abs error:", np.mean(np.abs(phi_t0_mid - phi_ic_expected_mid)))
print("max abs error :", np.max(np.abs(phi_t0_mid - phi_ic_expected_mid)))

ds.close()

time_whole = [0.     0.3125 0.625  0.9375 1.25   1.5625 1.875  2.1875 2.5    2.8125
 3.125  3.4375 3.75   4.0625 4.375  4.6875 5.    ]
nodal names = ['T', 'aux_sigma_af', 'u']

checking time = 0.3125

LEFT boundary (from phi.txt)
mean abs: 2.2461574293761486
min/max : 0.5 4.1840031633959045

RIGHT boundary (should be 0)
mean abs: 0.007692307692307693
max abs : 0.5
min/max : 0.0 0.5

BOTTOM boundary (should be 0.5)
mean abs error: 0.023809523809523808
max abs error : 0.5
min/max       : 0.0 0.5

TOP boundary (should be 0.5)
mean abs error: 0.0
max abs error : 0.0
min/max       : 0.5 0.5

IC check at t=0
mean abs error: 1.588472942925224e-16
max abs error : 8.881784197001252e-16


In [25]:
left_mask   = np.isclose(x, 0.0)
right_mask  = np.isclose(x, 0.019)
bottom_mask = np.isclose(y, 0.0)
top_mask    = np.isclose(y, 0.75)

corner_tol = 1e-12

right_no_corners = right_mask & (~np.isclose(y, 0.0, atol=corner_tol)) & (~np.isclose(y, 0.75, atol=corner_tol))
bottom_no_corners = bottom_mask & (~np.isclose(x, 0.0, atol=corner_tol)) & (~np.isclose(x, 0.019, atol=corner_tol))

k = 1

print("RIGHT no corners")
print("mean abs:", np.mean(np.abs(phi[k, right_no_corners])))
print("max abs :", np.max(np.abs(phi[k, right_no_corners])))
print("min/max :", phi[k, right_no_corners].min(), phi[k, right_no_corners].max())

print("\nBOTTOM no corners")
print("mean abs error:", np.mean(np.abs(phi[k, bottom_no_corners] - 0.5)))
print("max abs error :", np.max(np.abs(phi[k, bottom_no_corners] - 0.5)))
print("min/max       :", phi[k, bottom_no_corners].min(), phi[k, bottom_no_corners].max())

RIGHT no corners
mean abs: 0.0
max abs : 0.0
min/max : 0.0 0.0

BOTTOM no corners
mean abs error: 0.0
max abs error : 0.0
min/max       : 0.5 0.5


In [27]:
import netCDF4 as nc
import numpy as np

def unique_within_tolerance(arr, tol):
    arr = np.asarray(arr)
    sorted_arr = np.sort(arr)
    unique = [sorted_arr[0]]
    for i in range(1, len(sorted_arr)):
        if np.abs(sorted_arr[i] - unique[-1]) > tol:
            unique.append(sorted_arr[i])
    return np.array(unique)

def decode_name_array(arr):
    out = []
    arr = np.asarray(arr)
    if arr.dtype.kind in ("S", "U"):
        if arr.ndim == 2:
            for row in arr:
                s = b"".join(row).decode("utf-8", errors="ignore").strip() if row.dtype.kind == "S" else "".join(row).strip()
                out.append(s)
        else:
            out = [str(x).strip() for x in arr]
    else:
        for row in arr:
            chars = []
            for x in row:
                if isinstance(x, bytes):
                    chars.append(x.decode("utf-8", errors="ignore"))
                else:
                    chars.append(chr(x) if isinstance(x, (int, np.integer)) else str(x))
            out.append("".join(chars).strip())
    return out

def read_neu_to_np_correct(file_path):
    ds = nc.Dataset(file_path, "r")

    x_coords = ds.variables["coordx"][:]
    y_coords = ds.variables["coordy"][:]

    nod_names = decode_name_array(ds.variables["name_nod_var"][:])
    phi_idx = nod_names.index("u") + 1
    u = ds.variables[f"vals_nod_var{phi_idx}"][:]   # correct phi variable

    unique_x = unique_within_tolerance(np.array(x_coords), 1e-6)
    unique_y = unique_within_tolerance(np.array(y_coords), 1e-6)

    time_steps = u.shape[0]
    z_matrix = np.zeros((time_steps, len(unique_x), len(unique_y)))
    mask = np.zeros((len(unique_x), len(unique_y)))

    for i in range(len(x_coords)):
        xi = np.argmin(np.abs(x_coords[i] - unique_x))
        yi = np.argmin(np.abs(y_coords[i] - unique_y))
        mask[xi, yi] = 1
        z_matrix[:, xi, yi] = u[:, i]

    # nodal -> cell-centered
    z_matrix = (z_matrix[..., :-1] + z_matrix[..., 1:]) / 2.0
    z_matrix = (z_matrix[:, :-1, :] + z_matrix[:, 1:, :]) / 2.0

    ds.close()
    return z_matrix

f = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/solid_pinn_ref_out_sub_app0_exodus.e"
phi_from_exodus = read_neu_to_np_correct(f)

print("phi_from_exodus shape:", phi_from_exodus.shape)
print("phi_from_exodus min/max:", phi_from_exodus.min(), phi_from_exodus.max())
print("phi_from_exodus t0 min/max:", phi_from_exodus[0].min(), phi_from_exodus[0].max())
print("phi_from_exodus t1 min/max:", phi_from_exodus[1].min(), phi_from_exodus[1].max())

phi_from_exodus shape: (17, 20, 64)
phi_from_exodus min/max: 0.019286873502372324 10.483047280042607
phi_from_exodus t0 min/max: 0.05063449870848282 1.998796676955403
phi_from_exodus t1 min/max: 0.019286873502372324 4.233399349056073


In [28]:
import numpy as np

phi_true = phi_from_exodus
np.save("/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_phi_correct.npy", phi_true)
print(phi_true.shape, phi_true.min(), phi_true.max())

(17, 20, 64) 0.019286873502372324 10.483047280042607


In [29]:
import netCDF4 as nc
import numpy as np

def unique_within_tolerance(arr, tol):
    arr = np.asarray(arr)
    sorted_arr = np.sort(arr)
    unique = [sorted_arr[0]]
    for i in range(1, len(sorted_arr)):
        if np.abs(sorted_arr[i] - unique[-1]) > tol:
            unique.append(sorted_arr[i])
    return np.array(unique)

def decode_name_array(arr):
    out = []
    arr = np.asarray(arr)
    if arr.dtype.kind in ("S", "U"):
        if arr.ndim == 2:
            for row in arr:
                s = b"".join(row).decode("utf-8", errors="ignore").strip() if row.dtype.kind == "S" else "".join(row).strip()
                out.append(s)
        else:
            out = [str(x).strip() for x in arr]
    else:
        for row in arr:
            chars = []
            for x in row:
                if isinstance(x, bytes):
                    chars.append(x.decode("utf-8", errors="ignore"))
                else:
                    chars.append(chr(x) if isinstance(x, (int, np.integer)) else str(x))
            out.append("".join(chars).strip())
    return out

def read_neu_to_np_correct(file_path):
    ds = nc.Dataset(file_path, "r")

    nod_names = decode_name_array(ds.variables["name_nod_var"][:])
    phi_idx = nod_names.index("u") + 1
    phi_nodal = ds.variables[f"vals_nod_var{phi_idx}"][:]

    x_coords = ds.variables["coordx"][:]
    y_coords = ds.variables["coordy"][:]

    unique_x = unique_within_tolerance(np.array(x_coords), 1e-6)
    unique_y = unique_within_tolerance(np.array(y_coords), 1e-6)

    Nt = phi_nodal.shape[0]
    z = np.zeros((Nt, len(unique_x), len(unique_y)))

    for i in range(len(x_coords)):
        xi = np.argmin(np.abs(x_coords[i] - unique_x))
        yi = np.argmin(np.abs(y_coords[i] - unique_y))
        z[:, xi, yi] = phi_nodal[:, i]

    # nodal -> cell-centered
    z = 0.5 * (z[..., :-1] + z[..., 1:])
    z = 0.5 * (z[:, :-1, :] + z[:, 1:, :])

    ds.close()
    return z

phi_from_exodus = read_neu_to_np_correct(
    "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/solid_pinn_ref_out_sub_app0_exodus.e"
)

print("phi_from_exodus shape:", phi_from_exodus.shape)
print("phi_from_exodus min/max:", phi_from_exodus.min(), phi_from_exodus.max())
print("phi_from_exodus mean/std:", phi_from_exodus.mean(), phi_from_exodus.std())

phi_from_exodus shape: (17, 20, 64)
phi_from_exodus min/max: 0.019286873502372324 10.483047280042607
phi_from_exodus mean/std: 2.0318494436571943 1.688021063367896


In [30]:
phi_saved = np.load("/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_phi.npy")
if phi_saved.ndim == 4:
    phi_saved = phi_saved[0]

print("saved min/max:", phi_saved.min(), phi_saved.max())
print("diff mean abs:", np.mean(np.abs(phi_saved - phi_from_exodus)))
print("diff max abs :", np.max(np.abs(phi_saved - phi_from_exodus)))

saved min/max: -33.26056442378389 126.2331917444798
diff mean abs: 51.775022874388284
diff max abs : 125.6896620368025


In [31]:
# 1) IC at t=0
print(np.mean(np.abs(phi_t0_mid - phi_ic_expected_mid)))

# 2) right BC at t=0.3125, excluding corners
print(np.max(np.abs(phi[k, right_no_corners])))

# 3) bottom BC at t=0.3125, excluding corners
print(np.max(np.abs(phi[k, bottom_no_corners] - 0.5)))# 1) IC at t=0
print(np.mean(np.abs(phi_t0_mid - phi_ic_expected_mid)))

# 2) right BC at t=0.3125, excluding corners
print(np.max(np.abs(phi[k, right_no_corners])))

# 3) bottom BC at t=0.3125, excluding corners
print(np.max(np.abs(phi[k, bottom_no_corners] - 0.5)))

1.588472942925224e-16
0.0
0.0


In [32]:
print("phi_true shape:", phi_true.shape)
print("phi_true min :", phi_true.min())
print("phi_true max :", phi_true.max())
print("phi_true mean:", phi_true.mean())
print("phi_true std :", phi_true.std())

phi_true shape: (17, 20, 64)
phi_true min : 0.019286873502372324
phi_true max : 10.483047280042607
phi_true mean: 2.0318494436571943
phi_true std : 1.688021063367896


In [33]:
for k in [0, 1, 5, 10, 16]:
    print(f"t index {k}: min={phi_true[k].min():.6e}, max={phi_true[k].max():.6e}")

t index 0: min=5.063450e-02, max=1.998797e+00
t index 1: min=1.928687e-02, max=4.233399e+00
t index 5: min=2.993287e-02, max=5.923852e+00
t index 10: min=4.338490e-02, max=8.108662e+00
t index 16: min=4.744169e-02, max=1.048305e+01


In [34]:
import netCDF4 as nc
import numpy as np

def decode_name_array(arr):
    out = []
    arr = np.asarray(arr)
    if arr.dtype.kind in ("S", "U"):
        if arr.ndim == 2:
            for row in arr:
                s = b"".join(row).decode("utf-8", errors="ignore").strip() if row.dtype.kind == "S" else "".join(row).strip()
                out.append(s)
        else:
            out = [str(x).strip() for x in arr]
    else:
        for row in arr:
            chars = []
            for x in row:
                if isinstance(x, bytes):
                    chars.append(x.decode("utf-8", errors="ignore"))
                else:
                    chars.append(chr(x) if isinstance(x, (int, np.integer)) else str(x))
            out.append("".join(chars).strip())
    return out

f = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/solid_pinn_ref_out_sub_app0_exodus.e"
ds = nc.Dataset(f)

print("time_whole =", ds.variables["time_whole"][:])

nod_names = decode_name_array(ds.variables["name_nod_var"][:])
print("nodal names =", nod_names)

# phi is nodal variable named 'u'
phi_idx = nod_names.index("u") + 1
phi = ds.variables[f"vals_nod_var{phi_idx}"][:]

x = ds.variables["coordx"][:]
y = ds.variables["coordy"][:]
t = ds.variables["time_whole"][:]

# check BCs at non-initial time
k = 1   # t = 0.3125
print("\nchecking time =", t[k])

left_mask   = np.isclose(x, 0.0)
right_mask  = np.isclose(x, 0.019)
bottom_mask = np.isclose(y, 0.0)
top_mask    = np.isclose(y, 0.75)

print("\nLEFT boundary (from phi.txt)")
print("mean abs:", np.mean(np.abs(phi[k, left_mask])))
print("min/max :", phi[k, left_mask].min(), phi[k, left_mask].max())

print("\nRIGHT boundary (should be 0)")
print("mean abs:", np.mean(np.abs(phi[k, right_mask])))
print("max abs :", np.max(np.abs(phi[k, right_mask])))
print("min/max :", phi[k, right_mask].min(), phi[k, right_mask].max())

print("\nBOTTOM boundary (should be 0.5)")
print("mean abs error:", np.mean(np.abs(phi[k, bottom_mask] - 0.5)))
print("max abs error :", np.max(np.abs(phi[k, bottom_mask] - 0.5)))
print("min/max       :", phi[k, bottom_mask].min(), phi[k, bottom_mask].max())

print("\nTOP boundary (should be 0.5)")
print("mean abs error:", np.mean(np.abs(phi[k, top_mask] - 0.5)))
print("max abs error :", np.max(np.abs(phi[k, top_mask] - 0.5)))
print("min/max       :", phi[k, top_mask].min(), phi[k, top_mask].max())

# optional: check IC at t=0
yy = np.unique(np.round(y, 10))
yy.sort()
phi_ic_expected = 2.0 * np.cos(3.14 * (yy - 0.375) / 0.75)

# pick one interior x-line
x_unique = np.unique(np.round(x, 10))
x_unique.sort()
x_mid = x_unique[len(x_unique)//2]
mid_mask = np.isclose(x, x_mid)

yy_mid = y[mid_mask]
phi_t0_mid = phi[0, mid_mask]

order = np.argsort(yy_mid)
yy_mid = yy_mid[order]
phi_t0_mid = phi_t0_mid[order]

phi_ic_expected_mid = 2.0 * np.cos(3.14 * (yy_mid - 0.375) / 0.75)

print("\nIC check at t=0")
print("mean abs error:", np.mean(np.abs(phi_t0_mid - phi_ic_expected_mid)))
print("max abs error :", np.max(np.abs(phi_t0_mid - phi_ic_expected_mid)))

ds.close()

time_whole = [0.     0.3125 0.625  0.9375 1.25   1.5625 1.875  2.1875 2.5    2.8125
 3.125  3.4375 3.75   4.0625 4.375  4.6875 5.    ]
nodal names = ['T', 'aux_sigma_af', 'u']

checking time = 0.3125

LEFT boundary (from phi.txt)
mean abs: 2.2461574293761486
min/max : 0.5 4.1840031633959045

RIGHT boundary (should be 0)
mean abs: 0.007692307692307693
max abs : 0.5
min/max : 0.0 0.5

BOTTOM boundary (should be 0.5)
mean abs error: 0.023809523809523808
max abs error : 0.5
min/max       : 0.0 0.5

TOP boundary (should be 0.5)
mean abs error: 0.0
max abs error : 0.0
min/max       : 0.5 0.5

IC check at t=0
mean abs error: 1.588472942925224e-16
max abs error : 8.881784197001252e-16


In [35]:
import netCDF4 as nc
import numpy as np

def decode_name_array(arr):
    arr = np.asarray(arr)
    out = []
    if arr.dtype.kind in ("S", "U"):
        if arr.ndim == 2:
            for row in arr:
                s = b"".join(row).decode("utf-8", errors="ignore").strip() if row.dtype.kind == "S" else "".join(row).strip()
                out.append(s)
        else:
            out = [str(x).strip() for x in arr]
    else:
        for row in arr:
            chars = []
            for x in row:
                if isinstance(x, bytes):
                    chars.append(x.decode("utf-8", errors="ignore"))
                else:
                    chars.append(chr(x) if isinstance(x, (int, np.integer)) else str(x))
            out.append("".join(chars).strip())
    return out

f = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/solid_pinn_ref_out_sub_app0_sub_app0_exodus.e"
ds = nc.Dataset(f)

print("time_whole =", ds.variables["time_whole"][:])
print("element names =", decode_name_array(ds.variables["name_elem_var"][:]))

for k in ds.variables:
    print(k, ds.variables[k].dimensions, ds.variables[k].shape)

ds.close()

time_whole = [0.     0.3125 0.625  0.9375 1.25   1.5625 1.875  2.1875 2.5    2.8125
 3.125  3.4375 3.75   4.0625 4.375  4.6875 5.    ]
element names = ['T_fluid', 'flux', 'pressure', 'vel_x', 'vel_y']
time_whole ('time_step',) (17,)
eb_status ('num_el_blk',) (1,)
eb_prop1 ('num_el_blk',) (1,)
ns_status ('num_node_sets',) (4,)
ns_prop1 ('num_node_sets',) (4,)
ss_status ('num_side_sets',) (4,)
ss_prop1 ('num_side_sets',) (4,)
coordx ('num_nodes',) (845,)
coordy ('num_nodes',) (845,)
eb_names ('num_el_blk', 'len_name') (1, 256)
ns_names ('num_node_sets', 'len_name') (4, 256)
ss_names ('num_side_sets', 'len_name') (4, 256)
coor_names ('num_dim', 'len_name') (2, 256)
node_num_map ('num_nodes',) (845,)
connect1 ('num_el_in_blk1', 'num_nod_per_el1') (768, 4)
elem_num_map ('num_elem',) (768,)
elem_ss1 ('num_side_ss1',) (12,)
side_ss1 ('num_side_ss1',) (12,)
elem_ss2 ('num_side_ss2',) (64,)
side_ss2 ('num_side_ss2',) (64,)
elem_ss3 ('num_side_ss3',) (12,)
side_ss3 ('num_side_ss3',) (12,)
elem_s

MaskError: Cannot convert masked element to a Python int.